# Peer review with SciLogic

Before judging a paper, a reviewer has to reconstruct its argument: what the main conclusions are, what each one rests on, and where the chain is thin. A SciLogic encoding writes that reconstruction down, so it can be inspected, recomputed and questioned.

This example uses the encoding of **Zhuang, Chen and Carter (2017)**, who predict a D0<sub>19</sub>-ordered Mg<sub>3</sub>Al intermetallic compound with orbital-free and Kohn-Sham density functional theory and characterize its stability and ductility ([doi:10.1088/1361-651X/aa7e0c](https://doi.org/10.1088/1361-651X/aa7e0c)). The encoding is stored in the repository, so nothing here calls an LLM.

In [ ]:
import sys
from collections import Counter

import pandas as pd
import yaml
from IPython.display import Markdown

sys.path.insert(0, "../scripts")
import render

with open("../examples/zhuang2017_d019_mgal/zhuang2017_d019_mgal_claimchain.yaml") as f:
    chain = yaml.safe_load(f)

statements = {s["id"]: s for s in chain["statements"]}
S = lambda short_id: statements["zhuang2017:" + short_id]  # look up a statement by its short id

def show(sid):
    """Print a statement with its verdict, and every link into and out of it."""
    s = S(sid)
    a = s.get("assessment") or {}
    print(f"{sid} [{s['statement_type']}, {s.get('claim_role', '-')}] {s['text']}")
    if a:
        print(f"  verdict: {a['validity']}. {a['rationale']}")
    for r in chain["relations"]:
        if s["id"] in (r["subject"], r["object"]):
            strength = f", {r['support_strength'].lower()}" if r.get("support_strength") else ""
            print(f"  {render.short(r['subject'])} --{r['predicate']}--> {render.short(r['object'])}"
                  f"  ({r['asserted_by'].lower()}{strength})")

## The encoding at a glance

In [ ]:
types = Counter(s["statement_type"] for s in chain["statements"])
score, verdicts, _ = render.logic_score(chain)
checks = render.run_checks(chain)
obligations = Counter(po["obligation_status"] for s in chain["statements"]
                      for po in (s.get("assessment") or {}).get("proof_obligations", []))

print(f"Statements: {len(chain['statements'])} {dict(types)}")
print(f"Relations:  {len(chain['relations'])}, asserted by {dict(Counter(r['asserted_by'] for r in chain['relations']))}")
print(f"Verdicts:   {dict((k, v) for k, v in verdicts.items() if v)}  -> logic score {score:.2f}")
print(f"Checks:     {len(checks)} derived checks, {dict(Counter(c[2] for c in checks))}")
print(f"Proof obligations: {dict(obligations)}")

The one-page summary chart shows what the paper sets out to show, its main conclusions, and the verdict on each, in plain words.

In [ ]:
Markdown(f"```{{mermaid}}\n{render.summary_mermaid(chain)}```")

## Main claim C1: metastability

The paper's first main conclusion is that D0<sub>19</sub> Mg<sub>3</sub>Al is *at least metastable*. The encoding splits this into what is actually computed: energetic competitiveness (C1.1), dynamical stability (C1.2) and mechanical stability (C1.3), together with the paper's own counter-claim C1.4, that the phase is thermodynamically unstable against decomposition.

In [ ]:
show("C1")

In [ ]:
keep = {s["id"] for s in chain["statements"] if render.short(s["id"]).startswith(("C1", "H1"))}
Markdown(f"```{{mermaid}}\n{render.mermaid(chain, include=keep)}\n```")

C1.4 *qualifies* C1 rather than contradicting it: an unstable phase can still persist, but only if something stops it from decomposing. The paper offers that something only as a hypothesis, H1, a large kinetic barrier, and never computes it. H1 is therefore INCONCLUSIVE, and because C1 leans on it, C1 is PARTIAL.

In [ ]:
show("H1")

## The numbers behind the claims

Wherever the paper reports numbers, the encoding records them together with the criterion applied to them. `render.run_checks` recomputes all of these checks. For mechanical stability (C1.3), the Born criteria are evaluated from the paper's elastic constants:

In [ ]:
fmt = lambda x: str(x) if isinstance(x, bool) else f"{x:.4g}"
rows = []
for sid, check, status, computed in checks:
    expected = check.get("result", check.get("satisfied"))
    rows.append({"statement": render.short(sid), "check": check["expression"],
                 "paper": fmt(expected), "recomputed": fmt(computed), "status": status})
table = pd.DataFrame(rows)
table[table.statement.isin(["C1.3", "C2.1"])].style.hide(axis="index")

In [ ]:
print(f"{(table.status == 'ok').sum()} of {len(table)} derived checks reproduce from the recorded values.")

That every check recomputes shows that the recorded calculations are internally consistent. It does not validate the transcription of the values, the applicability of the criteria, or the conclusions drawn from them; those are still the reviewer's call.

## Main claim C2: ductility

The second main conclusion rests on four separate descriptors (C2.1 to C2.4), each with its own **warrant**, the inference rule that turns the number into evidence for ductility.

In [ ]:
show("C2")
print()
for sid in ["C2.1", "C2.2", "C2.3", "C2.4"]:
    w = S(sid).get("warrant") or next((r.get("warrant") for r in chain["relations"]
                                         if r["subject"] == S(sid)["id"] and r.get("warrant")), None)
    if w:
        print(f"{sid} warrant ({w.get('warrant_basis', '?').lower()}): {w['rule'].strip()}")

Two things stand out to a reviewer:

- The Pugh criterion behind C2.1 has the basis *empirical correlation*: a rule of thumb, not a law.
- C2.4 both **supports** and **undercuts** C2. C2 bundles two comparisons, D0<sub>19</sub> Mg<sub>3</sub>Al against hcp Mg and D0<sub>19</sub> against L1<sub>2</sub> Mg<sub>3</sub>Al; the anti-phase boundary energies help the first and hurt the second. The encoding exposes a compound claim that should be split.

## Proof obligations: what would make the claims hold?

Every assessed claim lists what would have to be shown, and which statements discharge each obligation. The open ones, with the data that would discharge them, are a ready-made list of requests to the authors.

In [ ]:
for s in chain["statements"]:
    for po in (s.get("assessment") or {}).get("proof_obligations", []):
        if po["obligation_status"] == "DISCHARGED":
            continue
        print(f"[{po.get('priority', '-')}] {render.short(s['id'])}: {po['text']}")
        for d in po.get("requested_data", []):
            how = f" ({d['method']})" if d.get("method") else ""
            print(f"      needs: {d['quantity']}{how}")

## From encoding to review report

Everything a reviewer needs to raise is now in the encoding. A few lines collect the candidate issues: the open obligations above, the annotator's concerns, warrants that rest on an empirical correlation or are unstated, and claims whose verdict the support links do not back up.

In [ ]:
issues = []
for s in chain["statements"]:
    sid = render.short(s["id"])
    for c in s.get("concerns", []):
        issues.append((sid, c["concern_type"].lower().replace("_", " "), c["description"]))
    w = s.get("warrant") or {}
    if w.get("warrant_basis") in {"EMPIRICAL_CORRELATION", "UNSTATED"}:
        issues.append((sid, "warrant", f"{w['warrant_basis'].lower()}: {w['rule'].strip()}"))
for r in chain["relations"]:
    w = r.get("warrant") or {}
    if w.get("warrant_basis") in {"EMPIRICAL_CORRELATION", "UNSTATED"}:
        issues.append((render.short(r["subject"]), "warrant", f"{w['warrant_basis'].lower()}: {w['rule'].strip()}"))
    if r["predicate"] in {"UNDERCUTS", "CONTRADICTS"}:
        issues.append((render.short(r["object"]), r["predicate"].lower(),
                       f"by {render.short(r['subject'])}: {r.get('note', '')}"))
_, disagreements = render.derived_score(chain)
for sid, annotator, rule in disagreements:
    issues.append((render.short(sid), "verdict check", f"annotator says {annotator}, the support links give {rule}"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(issues, columns=["statement", "issue", "detail"]).style.hide(axis="index")

The paper assessment sums it up:

In [ ]:
pa = chain["paper_assessment"]
print(f"Overall: {pa['overall_validity']}\n")
print(pa["summary"])
print("\nMissing evidence:")
for m in pa["missing_evidence"]:
    print(f"  - {m}")

## What this gives a reviewer

- **The argument, explicit.** Each main conclusion is broken into what is computed, what is assumed and what is only hypothesized, so the weak link (H1) is visible at once.
- **The numbers, recomputed.** All 22 derived checks are re-evaluated from the recorded values, so arithmetic slips would surface.
- **The open questions, concrete.** Open proof obligations name the data that would close them, which turns a vague "more evidence is needed" into specific requests.

The encoding is one annotator's reconstruction (here: hand-encoded, then converted and assessed with LLM assistance). It does not replace the reviewer's judgment; it makes that judgment faster to form and easier to check. Whether a domain expert would flag the same issues has not been tested.

To review a paper of your own: `claimchain paper.pdf -o paper.yaml`, then rerun this notebook on `paper.yaml`.